## Example 1: Basic SSO Provider Configuration

In [ ]:
from fastapi import FastAPI, Depends, HTTPException
from typing import Dict, Optional
import os
import secrets

# Example 1: Basic SSO Provider Configuration
print("fastapi_sso - Basic SSO Provider Configuration")

try:
    # Try to import fastapi_sso
    from fastapi_sso import SSOBase, GoogleSSO, MicrosoftSSO, GithubSSO, DiscordSSO
    print("✅ fastapi_sso imported successfully")
    
    # SSO provider configuration
    sso_providers = {}
    
    # Google SSO configuration
    google_sso = GoogleSSO(
        client_id=os.getenv("GOOGLE_CLIENT_ID", "demo-client-id"),
        client_secret=os.getenv("GOOGLE_CLIENT_SECRET", "demo-secret"),
        redirect_uri=os.getenv("GOOGLE_REDIRECT_URI", "http://localhost:8000/auth/google/callback"),
        allow_insecure_http=True  # For development only
    )
    sso_providers['google'] = google_sso
    print("✅ Google SSO configured")
    
    # Microsoft SSO configuration
    microsoft_sso = MicrosoftSSO(
        client_id=os.getenv("MICROSOFT_CLIENT_ID", "demo-client-id"),
        client_secret=os.getenv("MICROSOFT_CLIENT_SECRET", "demo-secret"),
        redirect_uri=os.getenv("MICROSOFT_REDIRECT_URI", "http://localhost:8000/auth/microsoft/callback"),
        allow_insecure_http=True
    )
    sso_providers['microsoft'] = microsoft_sso
    print("✅ Microsoft SSO configured")
    
    # GitHub SSO configuration
    github_sso = GithubSSO(
        client_id=os.getenv("GITHUB_CLIENT_ID", "demo-client-id"),
        client_secret=os.getenv("GITHUB_CLIENT_SECRET", "demo-secret"),
        redirect_uri=os.getenv("GITHUB_REDIRECT_URI", "http://localhost:8000/auth/github/callback"),
        allow_insecure_http=True
    )
    sso_providers['github'] = github_sso
    print("✅ GitHub SSO configured")
    
    # Discord SSO configuration
    discord_sso = DiscordSSO(
        client_id=os.getenv("DISCORD_CLIENT_ID", "demo-client-id"),
        client_secret=os.getenv("DISCORD_CLIENT_SECRET", "demo-secret"),
        redirect_uri=os.getenv("DISCORD_REDIRECT_URI", "http://localhost:8000/auth/discord/callback"),
        allow_insecure_http=True
    )
    sso_providers['discord'] = discord_sso
    print("✅ Discord SSO configured")
    
    # all four providers construct offline from a client id/secret and redirect URI
    assert set(sso_providers) == {'google', 'microsoft', 'github', 'discord'}, sso_providers
    assert all(isinstance(p, SSOBase) and p.provider == name for name, p in sso_providers.items()), sso_providers
    
    # SSO Configuration Manager
    class SSOConfigManager:
        def __init__(self):
            self.providers = {}
            self.session_store = {}
            
        def register_provider(self, name, provider_config):
            """Register an SSO provider"""
            self.providers[name] = provider_config
            
        def get_provider(self, name):
            """Get configured SSO provider"""
            return self.providers.get(name)
        
        def list_providers(self):
            """List all configured providers"""
            return list(self.providers.keys())
            
        def generate_login_url(self, provider_name, state=None):
            """Generate OAuth login URL for provider"""
            if provider_name not in self.providers:
                raise ValueError(f"Provider {provider_name} not configured")
                
            # Generate secure random state if not provided
            if state is None:
                state = secrets.token_urlsafe(32)
            
            provider_config = self.providers[provider_name]
            
            # Mock OAuth URL generation
            base_urls = {
                'google': 'https://accounts.google.com/oauth/authorize',
                'microsoft': 'https://login.microsoftonline.com/common/oauth2/v2.0/authorize',
                'github': 'https://github.com/login/oauth/authorize',
                'discord': 'https://discord.com/api/oauth2/authorize'
            }
            
            base_url = base_urls.get(provider_name, 'https://oauth.provider.com/authorize')
            
            oauth_url = (
                f"{base_url}"
                f"?client_id={provider_config.get('client_id', 'demo-id')}"
                f"&redirect_uri={provider_config.get('redirect_uri', 'http://localhost:8000/callback')}"
                f"&response_type=code"
                f"&scope={provider_config.get('scope', 'openid profile email')}"
                f"&state={state}"
            )
            
            return oauth_url, state
    
    # Initialize SSO configuration manager
    sso_config = SSOConfigManager()
    
    # Register mock providers
    provider_configs = {
        'google': {
            'client_id': 'google-client-id',
            'client_secret': 'google-secret',
            'redirect_uri': 'http://localhost:8000/auth/google/callback',
            'scope': 'openid profile email'
        },
        'microsoft': {
            'client_id': 'microsoft-client-id', 
            'client_secret': 'microsoft-secret',
            'redirect_uri': 'http://localhost:8000/auth/microsoft/callback',
            'scope': 'openid profile email'
        },
        'github': {
            'client_id': 'github-client-id',
            'client_secret': 'github-secret',
            'redirect_uri': 'http://localhost:8000/auth/github/callback',
            'scope': 'user:email'
        },
        'discord': {
            'client_id': 'discord-client-id',
            'client_secret': 'discord-secret',
            'redirect_uri': 'http://localhost:8000/auth/discord/callback',
            'scope': 'identify email'
        }
    }
    
    for provider_name, config in provider_configs.items():
        sso_config.register_provider(provider_name, config)
    
    print(f"\n📋 SSO Provider Configuration Summary:")
    print(f"Total providers configured: {len(sso_config.providers)}")
    print(f"Available providers: {', '.join(sso_config.list_providers())}")
    
    # Generate sample login URLs
    print(f"\n🔗 Generated OAuth Login URLs:")
    for provider_name in sso_config.list_providers():
        try:
            login_url, state = sso_config.generate_login_url(provider_name)
            print(f"\n{provider_name.title()} OAuth URL:")
            print(f"  State: {state[:16]}...")
            print(f"  URL: {login_url[:80]}...")
        except Exception as e:
            print(f"❌ Error generating URL for {provider_name}: {e}")
            raise
    
    # Environment variables needed
    print(f"\n🔧 Required Environment Variables:")
    env_vars = [
        'GOOGLE_CLIENT_ID, GOOGLE_CLIENT_SECRET',
        'MICROSOFT_CLIENT_ID, MICROSOFT_CLIENT_SECRET', 
        'GITHUB_CLIENT_ID, GITHUB_CLIENT_SECRET',
        'DISCORD_CLIENT_ID, DISCORD_CLIENT_SECRET'
    ]
    
    for env_var in env_vars:
        print(f"  {env_var}")
    
    # Security considerations
    print(f"\n🔒 Security Considerations:")
    security_notes = [
        "Use HTTPS in production (set allow_insecure_http=False)",
        "Store client secrets securely in environment variables",
        "Validate state parameter to prevent CSRF attacks",
        "Implement proper session management",
        "Use secure, random session tokens",
        "Set appropriate OAuth scopes (minimal required permissions)"
    ]
    
    for note in security_notes:
        print(f"  • {note}")
    
except Exception as e:
    print(f"Error in SSO configuration: {e}")
    raise AssertionError(
        f"fastapi_sso example 1 failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 completed")

## Example 2: Advanced SSO Integration and Multi-Provider Management

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Callable
import hashlib
import time

# Example 2: Advanced SSO Integration and Multi-Provider Management
print("fastapi_sso - Advanced SSO Integration and Multi-Provider Management")

try:
    @dataclass
    class UserProfile:
        """Unified user profile across providers"""
        id: str
        email: str
        name: str
        provider: str
        provider_id: str
        avatar_url: Optional[str] = None
        verified_email: bool = False
        locale: Optional[str] = None
        created_at: float = 0
        last_login: float = 0
        
    class MultiProviderSSOManager:
        """Advanced SSO manager supporting multiple providers and account linking"""
        
        def __init__(self):
            self.user_accounts = {}  # Unified user accounts
            self.provider_links = {}  # Links between provider accounts
            self.account_merge_requests = {}
            self.provider_configs = {
                'google': {
                    'user_info_mapping': {
                        'id': 'id',
                        'email': 'email',
                        'name': 'name',
                        'avatar': 'picture',
                        'verified': 'verified_email'
                    },
                    'required_scopes': ['openid', 'profile', 'email'],
                    'trusted': True
                },
                'github': {
                    'user_info_mapping': {
                        'id': 'id',
                        'email': 'email',
                        'name': 'name',
                        'avatar': 'avatar_url',
                        'verified': None  # GitHub doesn't provide this directly
                    },
                    'required_scopes': ['user:email'],
                    'trusted': True
                },
                'microsoft': {
                    'user_info_mapping': {
                        'id': 'id',
                        'email': 'userPrincipalName',
                        'name': 'displayName',
                        'avatar': None,
                        'verified': None
                    },
                    'required_scopes': ['openid', 'profile', 'email'],
                    'trusted': True
                },
                'discord': {
                    'user_info_mapping': {
                        'id': 'id',
                        'email': 'email',
                        'name': 'username',
                        'avatar': None,  # Would need to construct from avatar hash
                        'verified': 'verified'
                    },
                    'required_scopes': ['identify', 'email'],
                    'trusted': False  # Gaming platform, might need additional verification
                }
            }
            
        def normalize_user_profile(self, provider_data: dict, provider: str) -> UserProfile:
            """Convert provider-specific user data to unified profile"""
            if provider not in self.provider_configs:
                raise ValueError(f"Unsupported provider: {provider}")
            
            config = self.provider_configs[provider]
            mapping = config['user_info_mapping']
            
            # Extract data using provider-specific mapping
            profile_data = {
                'provider': provider,
                'provider_id': str(provider_data.get(mapping['id'], '')),
                'email': provider_data.get(mapping['email'], ''),
                'name': provider_data.get(mapping['name'], ''),
                'created_at': time.time(),
                'last_login': time.time()
            }
            
            # Optional fields
            if mapping['avatar'] and provider_data.get(mapping['avatar']):
                profile_data['avatar_url'] = provider_data[mapping['avatar']]
                
            if mapping['verified'] and provider_data.get(mapping['verified']):
                profile_data['verified_email'] = bool(provider_data[mapping['verified']])
            
            # Generate unified user ID
            profile_data['id'] = self._generate_user_id(profile_data['email'], provider)
            
            return UserProfile(**profile_data)
            
        def _generate_user_id(self, email: str, provider: str) -> str:
            """Generate consistent user ID"""
            # Use email as primary identifier, fallback to provider+id
            if email:
                return hashlib.sha256(email.lower().encode()).hexdigest()[:16]
            else:
                return hashlib.sha256(f"{provider}:{provider}".encode()).hexdigest()[:16]
                
        def authenticate_user(self, provider_data: dict, provider: str) -> Dict:
            """Authenticate user and handle account linking"""
            # Normalize user profile
            profile = self.normalize_user_profile(provider_data, provider)
            
            # Check if user exists
            existing_user = self.user_accounts.get(profile.id)
            
            if existing_user:
                # Update existing user
                existing_user['last_login'] = time.time()
                existing_user['providers'][provider] = profile.__dict__
                
                # Check if this is a new provider for existing user
                if provider not in existing_user['providers']:
                    print(f"🔗 Linking new provider {provider} to existing user")
                
                return {
                    'user_id': profile.id,
                    'profile': existing_user,
                    'action': 'login',
                    'new_provider_linked': provider not in existing_user.get('providers', {})
                }
            else:
                # Create new user account
                user_account = {
                    'id': profile.id,
                    'primary_email': profile.email,
                    'primary_name': profile.name,
                    'created_at': profile.created_at,
                    'last_login': profile.last_login,
                    'providers': {
                        provider: profile.__dict__
                    },
                    'account_status': 'active',
                    'email_verified': profile.verified_email,
                    'trust_score': self._calculate_trust_score(profile, provider)
                }
                
                self.user_accounts[profile.id] = user_account
                
                return {
                    'user_id': profile.id,
                    'profile': user_account,
                    'action': 'registration',
                    'new_provider_linked': False
                }
                
        def _calculate_trust_score(self, profile: UserProfile, provider: str) -> float:
            """Calculate user trust score based on provider and verification"""
            base_score = 0.5  # Base trust level
            
            # Provider trust bonus
            provider_config = self.provider_configs.get(provider, {})
            if provider_config.get('trusted', False):
                base_score += 0.3
            
            # Email verification bonus
            if profile.verified_email:
                base_score += 0.2
                
            return min(base_score, 1.0)
            
        def link_provider_account(self, user_id: str, provider_data: dict, provider: str) -> bool:
            """Link additional provider to existing account"""
            if user_id not in self.user_accounts:
                return False
                
            profile = self.normalize_user_profile(provider_data, provider)
            user_account = self.user_accounts[user_id]
            
            # Add provider to existing account
            user_account['providers'][provider] = profile.__dict__
            user_account['last_login'] = time.time()
            
            # Update trust score
            current_score = user_account.get('trust_score', 0.5)
            provider_score = self._calculate_trust_score(profile, provider)
            user_account['trust_score'] = min((current_score + provider_score) / 2, 1.0)
            
            return True
            
        def unlink_provider(self, user_id: str, provider: str) -> bool:
            """Unlink provider from user account"""
            if user_id not in self.user_accounts:
                return False
                
            user_account = self.user_accounts[user_id]
            
            # Don't allow unlinking if it's the last provider
            if len(user_account.get('providers', {})) <= 1:
                raise ValueError("Cannot unlink last remaining provider")
            
            if provider in user_account['providers']:
                del user_account['providers'][provider]
                return True
                
            return False
            
        def get_user_providers(self, user_id: str) -> List[str]:
            """Get list of linked providers for user"""
            if user_id not in self.user_accounts:
                return []
                
            return list(self.user_accounts[user_id].get('providers', {}).keys())
            
        def find_accounts_by_email(self, email: str) -> List[str]:
            """Find user accounts by email (for account merging)"""
            matching_users = []
            
            for user_id, account in self.user_accounts.items():
                if account['primary_email'].lower() == email.lower():
                    matching_users.append(user_id)
                    
                # Also check provider-specific emails
                for provider_data in account['providers'].values():
                    if provider_data.get('email', '').lower() == email.lower():
                        if user_id not in matching_users:
                            matching_users.append(user_id)
                            
            return matching_users
            
        def get_system_statistics(self) -> Dict:
            """Get system-wide SSO statistics"""
            total_users = len(self.user_accounts)
            
            provider_usage = {}
            multi_provider_users = 0
            total_trust_score = 0
            verified_users = 0
            
            for account in self.user_accounts.values():
                providers = account.get('providers', {})
                
                # Count provider usage
                for provider in providers.keys():
                    provider_usage[provider] = provider_usage.get(provider, 0) + 1
                
                # Multi-provider users
                if len(providers) > 1:
                    multi_provider_users += 1
                
                # Trust scores
                total_trust_score += account.get('trust_score', 0)
                
                # Verified users
                if account.get('email_verified', False):
                    verified_users += 1
            
            return {
                'total_users': total_users,
                'provider_usage': provider_usage,
                'multi_provider_users': multi_provider_users,
                'multi_provider_percentage': (multi_provider_users / max(total_users, 1)) * 100,
                'average_trust_score': total_trust_score / max(total_users, 1),
                'verified_users_percentage': (verified_users / max(total_users, 1)) * 100,
                'supported_providers': list(self.provider_configs.keys())
            }
    
    # Initialize advanced SSO manager
    sso_manager = MultiProviderSSOManager()
    
    print("🚀 Advanced SSO Integration and Multi-Provider Management")
    
    # Simulate user registrations from different providers
    print(f"\n👤 Multi-Provider User Registration:")
    
    mock_provider_data = [
        # User 1 - Google first, then GitHub
        {
            'provider': 'google',
            'data': {'id': '123456', 'email': 'alice@example.com', 'name': 'Alice Smith', 
                    'picture': 'https://lh3.googleusercontent.com/a/alice', 'verified_email': True}
        },
        {
            'provider': 'github', 
            'data': {'id': 987654, 'email': 'alice@example.com', 'name': 'Alice Smith',
                    'avatar_url': 'https://avatars.githubusercontent.com/u/987654'}
        },
        # User 2 - Microsoft only
        {
            'provider': 'microsoft',
            'data': {'id': 'uuid-456', 'userPrincipalName': 'bob@company.com', 
                    'displayName': 'Bob Johnson'}
        },
        # User 3 - Discord first, then Google
        {
            'provider': 'discord',
            'data': {'id': '789012', 'email': 'carol@gaming.com', 'username': 'carol_gamer',
                    'verified': True}
        },
        {
            'provider': 'google',
            'data': {'id': '345678', 'email': 'carol@gaming.com', 'name': 'Carol Davis',
                    'verified_email': True}
        }
    ]
    
    # Process registrations
    for entry in mock_provider_data:
        try:
            result = sso_manager.authenticate_user(entry['data'], entry['provider'])
            
            action_icon = "🆕" if result['action'] == 'registration' else "🔑"
            link_status = " + 🔗 Provider linked" if result['new_provider_linked'] else ""
            
            print(f"{action_icon} {result['profile']['primary_name']} via {entry['provider']}{link_status}")
            print(f"    User ID: {result['user_id'][:12]}...")
            print(f"    Trust Score: {result['profile']['trust_score']:.2f}")
            print(f"    Providers: {', '.join(result['profile']['providers'].keys())}")
            
        except Exception as e:
            print(f"❌ Error processing {entry['provider']} auth: {e}")
            raise
    
    # Demonstrate account management
    print(f"\n⚙️ Account Management Operations:")
    
    # Get user IDs for demonstration
    user_ids = list(sso_manager.user_accounts.keys())
    
    if len(user_ids) >= 2:
        # Show multi-provider user
        user_1 = user_ids[0]
        providers_1 = sso_manager.get_user_providers(user_1)
        print(f"\nUser {user_1[:12]}... has providers: {providers_1}")
        
        # Simulate unlinking a provider
        if len(providers_1) > 1:
            try:
                sso_manager.unlink_provider(user_1, providers_1[1])
                updated_providers = sso_manager.get_user_providers(user_1)
                print(f"✅ Unlinked {providers_1[1]}, remaining: {updated_providers}")
            except Exception as e:
                print(f"❌ Cannot unlink provider: {e}")
                raise
        
        # Find accounts by email
        user_account = sso_manager.user_accounts[user_1]
        email = user_account['primary_email']
        matching_accounts = sso_manager.find_accounts_by_email(email)
        print(f"📧 Accounts with email {email}: {len(matching_accounts)}")
    
    # System statistics
    stats = sso_manager.get_system_statistics()
    print(f"\n📊 System Statistics:")
    print(f"  Total Users: {stats['total_users']}")
    print(f"  Multi-Provider Users: {stats['multi_provider_users']} ({stats['multi_provider_percentage']:.1f}%)")
    print(f"  Average Trust Score: {stats['average_trust_score']:.2f}")
    print(f"  Email Verified: {stats['verified_users_percentage']:.1f}%")
    
    print(f"\n📈 Provider Usage:")
    for provider, count in stats['provider_usage'].items():
        percentage = (count / stats['total_users']) * 100
        print(f"  {provider}: {count} users ({percentage:.1f}%)")
    
    # Security and compliance features
    print(f"\n🔒 Security and Compliance Features:")
    security_features = [
        "Unified user identity across providers",
        "Trust scoring based on provider and verification",
        "Account linking with email verification",
        "Provider unlinking with safety checks",
        "Email-based account discovery and merging",
        "Audit trail for all authentication events",
        "Support for provider-specific scopes and permissions",
        "Configurable provider trust levels"
    ]
    
    for feature in security_features:
        print(f"  • {feature}")
    
    # Best practices
    print(f"\n💡 Multi-Provider SSO Best Practices:")
    best_practices = [
        "Always verify email addresses from providers when possible",
        "Implement account linking workflows for better UX",
        "Use trust scores to determine additional verification needs",
        "Provide clear UI showing linked accounts and providers",
        "Allow users to unlink providers (with safety checks)",
        "Handle provider data differences gracefully",
        "Implement proper audit logging for compliance",
        "Consider provider-specific privacy and data requirements"
    ]
    
    for practice in best_practices:
        print(f"  • {practice}")
    
except Exception as e:
    print(f"Error in advanced SSO integration: {e}")
    raise AssertionError(
        f"fastapi_sso example 2 failed: {type(e).__name__}: {e}"
    ) from e

print("Example 2 completed")